In [10]:
!pip install openai
!pip install kubernetes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 50.5 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 140.2 kB/s eta 0:00:0000:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.6/806.6 kB 2.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.1/231.1 kB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.3/246.3 kB 1.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 622.7 kB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 1.2 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: pyyaml
    Found existing installation: PyYAML 6.0.1
    Uninstalling PyYAML-6.0.1:
      Successfully uninstalled PyYAML-6.0.1


In [8]:
import requests
import os

# Try to connect to Ollama using common Docker addresses
ollama_urls = [
    #"http://ollama:11434",                  # If Ollama is in a container named 'ollama'
    "http://host.docker.internal:11434",    # If Ollama is on the host machine
]

payload = {
    "model": "llama3.2",
    "prompt": "Hello from my agent",
    "stream": False
}

for url in ollama_urls:
    try:
        r = requests.post(f"{url}/api/generate", json=payload)
        print(f"Successfully connected to Ollama at {url}")
        print(r.json()["response"])
        break
    except requests.exceptions.ConnectionError:
        print(f"Could not connect to Ollama at {url}")

Successfully connected to Ollama at http://host.docker.internal:11434
I don't have any information about you, so I'm not sure who your agent is. Can you please tell me what's going on or what kind of assistance you need? I'll do my best to help!


🤖 Kubernetes Environment Agent
🔍 Checking Kubernetes setup...
✅ kubectl: 
❌ Cannot access cluster: E0706 14:48:47.950138    7384 memcache.go:265] "Unhandled Error" err="couldn't get current server API group list: Get \"http://localhost:8080/api?timeout=32s\": dial tcp 127.0.0.1:8080: connect: connection refused"
E0706 14:48:47.951129    7384 memcache.go:265] "Unhandled Error" err="couldn't get current server API group list: Get \"http://localhost:8080/api?timeout=32s\": dial tcp 127.0.0.1:8080: connect: connection refused"
E0706 14:48:47.953313    7384 memcache.go:265] "Unhandled Error" err="couldn't get current server API group list: Get \"http://localhost:8080/api?timeout=32s\": dial tcp 127.0.0.1:8080: connect: connection refused"
E0706 14:48:47.954474    7384 memcache.go:265] "Unhandled Error" err="couldn't get current server API group list: Get \"http://localhost:8080/api?timeout=32s\": dial tcp 127.0.0.1:8080: connect: connection refused"
E0706 14:48:47.955132    7384 memcache.go

# Get namespace and pod name from user inputs.

In [16]:
import requests
import json

def ask_llama(prompt, model="llama3.2"):
    """
    Send a question to Llama 3.2 running on Ollama
    
    Args:
        prompt (str): The question/prompt to ask
        model (str): Model name (default: llama3.2)
    
    Returns:
        str: Response from the model
    """
    url = "http://host.docker.internal:11434/api/generate"
    
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False  # Set to True for streaming responses
    }
    
    try:
        response = requests.post(url, json=payload)
        response.raise_for_status()
        
        # Parse the response
        result = response.json()
        return result.get("response", "No response received")
    
    except requests.exceptions.ConnectionError:
        return "Error: Cannot connect to Ollama. Make sure it's running on http://localhost:11434/"
    except requests.exceptions.RequestException as e:
        return f"Error: {str(e)}"

# Example usage
if __name__ == "__main__":
    question = "provide only Namespace: ? and Pod_Name: ? from following inputs:  \
                namespace prod pod api-gateway-7d5c8f9b7-abc12"
    response = ask_llama(question)
    print(f"Question: {question}")
    print(f"Response: \n {response}")

Question: provide only Namespace: ? and Pod_Name: ? from following inputs:                  namespace prod pod api-gateway-7d5c8f9b7-abc12
Response: 
 Namespace: prod
Pod_Name: api-gateway-7d5c8f9b7-abc12


# Agent execution from responce

In [15]:
import requests
import json
import re

def select_agent_with_llm(response_text):
    """
    Use LLM to select the appropriate agent based on the response context
    """
    # Step 1 response from your LLM execution
    step1_response = response_text
    
    # Prepare the prompt for the LLM to select agent
    print("Step 1 responce", step1_response)
    
    prompt = f"""
    Context: {step1_response}

    AVAILABLE AGENTS (choose exactly one):
    1. agent_k8s_log - For Kubernetes operations, pods, containers, logs, debugging
    2. agent_upload - For file uploads, storage, data transfer
    3. agent_sent - For sending notifications, alerts, emails
    
    IMPORTANT RULES:
    - If you see pod names, namespaces, or any Kubernetes terms → ALWAYS choose agent_k8s_log

    YOUR RESPONSE MUST BE EXACTLY ONE OF THESE THREE STRINGS:
    agent_k8s_log
    agent_upload
    agent_sent

    Do NOT add any explanation, punctuation, or extra text. Just the agent name.
    """
    
    # Call Ollama API with llama3.2 model
    ollama_url = "http://host.docker.internal:11434/api/generate"
    
    payload = {
        "model": "llama3.2",
        "prompt": prompt,
        "stream": False,
        "temperature": 0.3  # Lower temperature for more deterministic output
    }
    
    try:
        response = requests.post(ollama_url, json=payload)
        response.raise_for_status()
        
        result = response.json()
        print("Response Selection", result)
        
        selected_agent = result.get('response', '').strip().lower()
        
        # Clean up the response to get just the agent name
        # Remove any extra text, quotes, or formatting
        print("Before Selected Agent by LLM:", selected_agent)
        selected_agent = re.sub(r'[^a-z0-9_]', '', selected_agent)
        print("After Selected Agent by LLM:", selected_agent)
        return selected_agent
        
    except requests.exceptions.RequestException as e:
        print(f"Error calling Ollama API: {e}")

# Main execution
if __name__ == "__main__":
    # Step 1 response from LLM execution
    step1_response = """
    Namespace: prod
    Pod Name: api-gateway-7d5c8f9b7-abc12
    """
    
    # Select the appropriate agent
    selected_agent = select_agent_with_llm(step1_response)
    
    print(f"Selected Agent: {selected_agent}")

Step 1 responce 
    Namespace: prod
    Pod Name: api-gateway-7d5c8f9b7-abc12
    
Response Selection {'model': 'llama3.2', 'created_at': '2026-07-13T05:48:45.347117482Z', 'response': 'agent_k8s_log', 'done': True, 'done_reason': 'stop', 'context': [128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 271, 128009, 128006, 882, 128007, 1432, 262, 9805, 25, 720, 262, 43062, 25, 14814, 198, 262, 17241, 4076, 25, 6464, 2427, 12312, 12, 22, 67, 20, 66, 23, 69, 24, 65, 22, 12, 13997, 717, 88179, 262, 68819, 15432, 44978, 320, 28150, 7041, 832, 997, 262, 220, 16, 13, 8479, 4803, 23, 82, 5337, 482, 1789, 67474, 7677, 11, 55687, 11, 24794, 11, 18929, 11, 28803, 198, 262, 220, 17, 13, 8479, 22433, 482, 1789, 1052, 67663, 11, 5942, 11, 828, 8481, 198, 262, 220, 18, 13, 8479, 25084, 482, 1789, 11889, 22736, 11, 30350, 11, 14633, 7361, 262, 68240, 44897, 50, 512, 262, 482, 1442, 499, 1518, 7661, 5144, 11, 59191, 11, 477, 904, 67474, 3878, 11651, 68514, 5268, 8479, 4803, 23